# BERT for Sentiment Analysis => fine-tuning a pre-trained BERT

In [1]:
# %pip install transformers -U

In [2]:
import transformers
import tensorflow as tf

c:\Users\milos\nlp-document-classification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Downloading large review movie dataset (25000 reviews)

In [3]:
# !wget https://thome.isir.upmc.fr/classes/RITAL/json_pol.json

In [4]:
import json
from collections import Counter

# Loading json
file = "./json_pol.json"
with open(file, encoding="utf-8") as f:
    data = json.load(f)


# Quick Check
counter = Counter((x[1] for x in data))
print("Number of reviews : ", len(data))
print("----> # of positive : ", counter[1])
print("----> # of negative : ", counter[0])
print("")
print(data[0])


Number of reviews :  25000
----> # of positive :  12500
----> # of negative :  12500

['Although credit should have been given to Dr. Seuess for stealing the story-line of "Horton Hatches The Egg", this was a fine film. It touched both the emotions and the intellect. Due especially to the incredible performance of seven year old Justin Henry and a script that was sympathetic to each character (and each one\'s predicament), the thought provoking elements linger long after the tear jerking ones are over. Overall, superior acting from a solid cast, excellent directing, and a very powerful script. The right touches of humor throughout help keep a "heavy" subject from becoming tedious or difficult to sit through. Lastly, this film stands the test of time and seems in no way dated, decades after it was released.', 1]


# Getting the Tokenizer

In [5]:
# from transformers import DistilBertForSequenceClassification, DistilBertTokenizer
# tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-cased')
model_name = "haisongzhang/roberta-tiny-cased"
# model_name = "bert-base-cased"

# model_name = "distilbert-base-uncased-finetuned-yelp-polarity"
# model_name = "textattack/bert-base-uncased-yelp-polarity"
# model_name = "rttl-ai/bert-base-uncased-yelp-reviews"


from transformers import AutoTokenizer  # , BertForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained(model_name)


# Experiment the Tokenizer on the first train review

In [6]:
maxL = 512  # Max length of the sequence

string_tokenized = (
    tokenizer(  # encode_plus I think it deprecated I replaced it with _encode_plus
        data[0][0],
        return_tensors="pt",
        add_special_tokens=True,  # add '[CLS]' and '[SEP]'
        max_length=maxL,  # set max length
        truncation=True,  # truncate longer messages
        # pad_to_max_length=True,
        padding="max_length",  # add padding
        return_attention_mask=True,
    )
)

The output of the tokenizer string_tokenized (class BatchEncoding) returns two elements:


*   string_tokenized['input_ids']: the index of each token in the dictionary
*   string_tokenized['attention_mask']: a binary mask (0 to ignore the token, 1 to consider it). This is because we need tensor a fixed length and we have reviews with a variable number of words



In [7]:
print(string_tokenized["input_ids"])
print(string_tokenized["attention_mask"])

tensor([[  101,  1780,  4755,  1431,  1138,  1151,  1549,  1106,   173,  1197,
           119, 14516, 10589,  1116,  1111, 11569,  1103,  1642,   118,  1413,
          1104,   107, 16358, 11530, 15524,  1279,  1103,  9069,   107,   117,
          1142,  1108,   170,  2503,  1273,   119,  1122,  4270,  1241,  1103,
          6288,  1105,  1103,  1107,  7854, 18465,   119,  1496,  2108,  1106,
          1103, 10965,  2099,  1104,  1978,  1214,  1385,  1198,  1394,  1119,
          1179,  1616,  1105,   170,  5444,  1115,  1108, 13493,  1106,  1296,
          1959,   113,  1105,  1296,  1141,   112,   188,  3073, 13328, 11462,
           114,   117,  1103,  1354,  5250, 20202,  3050,   181,  7728,  1263,
          1170,  1103,  7591, 26525,  3200,  1132,  1166,   119,  2905,   117,
          7298,  3176,  1121,   170,  4600,  2641,   117,  6548, 10404,   117,
          1105,   170,  1304,  3110,  5444,   119,  1103,  1268, 13193,  1104,
          8594,  2032,  1494,  1712,   170,   107,  

**You can use the BERT model for directly predicting polarity.** Let us apply that on the first review which has been tokenized with string_tokenized.

# Let's tokenize the whole dataset

In [8]:
import numpy as np

maxL = 512


inputs_tokens = []
attention_masks = []

for i in range(len(data)):
    if i % 2500 == 0:
        print(i)
    string_tokenized = tokenizer(
        data[i][0],
        return_tensors="pt",
        add_special_tokens=True,  # add '[CLS]' and '[SEP]'
        max_length=maxL,  # set max length
        truncation=True,  # truncate longer messages
        # pad_to_max_length=True,
        padding="max_length",  # add padding
        return_attention_mask=True,
    )

    inputs_tokens.append(string_tokenized["input_ids"])
    attention_masks.append(string_tokenized["attention_mask"])

0
2500
5000
7500
10000
12500
15000
17500
20000
22500


# Let's create a 'TensorDataSet' FOR THE SAMPLES where each element is a triplet composed of token word index, token mask, and label

In [9]:
# print(data[1300][1])
# print(y[0])

In [10]:
inputs_tokens[1].shape

torch.Size([1, 512])

In [11]:
import torch

# Converting input tokens to torch tensors
inputs_tokens = torch.cat(inputs_tokens, dim=0)
attention_masks = torch.cat(attention_masks, dim=0)


In [12]:
# Converting labels to torch tensor
# y = torch.zeros((len(data),2), dtype=torch.float)
y = torch.zeros((len(data),), dtype=torch.float)

for i in range(len(data)):
    y[i] = data[i][1]
    # y[i][data[i][1]] = 1
# y = torch.from_numpy(y)

In [13]:
print(inputs_tokens.shape)
print(attention_masks.shape)
print(y.shape)

torch.Size([25000, 512])
torch.Size([25000, 512])
torch.Size([25000])


In [14]:
from sklearn.model_selection import train_test_split

np.random.seed(0)
rs = 10

(
    inputs_tokens_train,
    inputs_tokens_test,
    attention_masks_train,
    attention_masks_test,
    y_train,
    y_test,
) = train_test_split(inputs_tokens, attention_masks, y, test_size=0.5, random_state=rs)

print(inputs_tokens_train.shape)
print(inputs_tokens_test.shape)

print(attention_masks_train.shape)
print(attention_masks_test.shape)

print(y_train.shape)
print(y_test.shape)

torch.Size([12500, 512])
torch.Size([12500, 512])
torch.Size([12500, 512])
torch.Size([12500, 512])
torch.Size([12500])
torch.Size([12500])


In [15]:
# print(y_train[10000])

In [16]:
from torch.utils.data import (
    TensorDataset,
    random_split,
    DataLoader,
    RandomSampler,
    SequentialSampler,
)

dataset_train = TensorDataset(inputs_tokens_train, attention_masks_train, y_train)
dataset_test = TensorDataset(inputs_tokens_test, attention_masks_test, y_test)

# Lets download a BERT model for word embedding

In [17]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(model_name)

Loading weights: 100%|██████████| 71/71 [00:00<00:00, 20503.69it/s]
BertForSequenceClassification LOAD REPORT from: haisongzhang/roberta-tiny-cased
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [18]:
print(model)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 512, padding_idx=0)
      (position_embeddings): Embedding(512, 512)
      (token_type_embeddings): Embedding(2, 512)
      (LayerNorm): LayerNorm((512,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-3): 4 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=512, out_features=512, bias=True)
              (key): Linear(in_features=512, out_features=512, bias=True)
              (value): Linear(in_features=512, out_features=512, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=512, out_features=512, bias=True)
              (LayerNorm): LayerNorm((512,), eps=1e-12, e

In [19]:
# train_dataloader = DataLoader(dataset_train, batch_size=2,shuffle=True)

# it, tm, labelsb = next(iter(train_dataloader))
#
# pred = model(input_ids=it, attention_mask=tm)
# print(pred[0])

In [20]:
# print(pred.logits)

# FINE-TUNING THE MODEL

In [21]:
import gc

gc.collect()
torch.cuda.empty_cache()

In [22]:
def accuracy(model, dataloader):
    model.eval()
    nbgood = 0
    for idx, batch in enumerate(dataloader):
        b_input_ids = batch[0].cuda()
        b_input_mask = batch[1].cuda()
        b_labels = batch[2].cuda()

        with torch.no_grad():
            pred = model(input_ids=b_input_ids, attention_mask=b_input_mask)
            yhat = pred.logits.argmax(axis=1)
            ytrue = b_labels
            nbgood += (yhat == ytrue).sum()

        if idx > 0 and idx % 250 == 0:
            print("idx=", idx, " nbgood=", nbgood.item())

    acc = nbgood / 125.0
    return acc.item()


In [23]:
def accuracy_batch(model, b_input_ids, b_input_mask, b_labels):
    model.eval()

    with torch.no_grad():
        pred = model(input_ids=b_input_ids, attention_mask=b_input_mask)
        yhat = pred.logits.argmax(axis=1)
        ytrue = b_labels

        print(yhat.detach().cpu().numpy())
        print(ytrue.detach().cpu().numpy())

        acc = (yhat == ytrue).sum() * 4.0

    return acc.item()


In [24]:
# create DataLoaders with samplers

import torch.nn as nn
import torch.optim as optim

tb = int(25)
train_dataloader = DataLoader(dataset_train, batch_size=tb, shuffle=True)
test_dataloader = DataLoader(dataset_test, batch_size=tb, shuffle=False)
nbTrain = len(data)

temb = 768
temb = 512
# features = np.zeros((nbTrain, temb))
# nbtach = int(nbTrain/tb)
# print(f"nb batches={nbtach}")

# nbgood =torch.tensor(0.0).cuda()
nbepochs = 5
loss = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# Comuting CLS features
model.train()
model.cuda()

for e in range(nbepochs):
    for idx, batch in enumerate(train_dataloader):
        b_input_ids = batch[0].cuda()
        b_input_mask = batch[1].cuda()
        b_labels = batch[2].cuda().long()

        optimizer.zero_grad()

        pred = model(input_ids=b_input_ids, attention_mask=b_input_mask)
        # print(b_labels)
        ce = loss(pred.logits, b_labels)
        ce.backward()
        optimizer.step()

        if idx > 0 and idx % 250 == 0:
            print(
                "** batch: ",
                idx,
                " acc train=",
                accuracy(model, train_dataloader),
                " acc test=",
                accuracy(model, test_dataloader),
            )

    print(
        "**** epoch",
        (e + 1),
        " acc train=",
        accuracy(model, train_dataloader),
        " acc test=",
        accuracy(model, test_dataloader),
    )


idx= 250  nbgood= 5592
idx= 250  nbgood= 5402
** batch:  250  acc train= 88.97600555419922  acc test= 86.14400482177734
idx= 250  nbgood= 5980
idx= 250  nbgood= 5621
**** epoch 1  acc train= 95.07200622558594  acc test= 89.61600494384766
idx= 250  nbgood= 6080
idx= 250  nbgood= 5619
** batch:  250  acc train= 96.71200561523438  acc test= 89.66400146484375
idx= 250  nbgood= 6067
idx= 250  nbgood= 5513
**** epoch 2  acc train= 96.69600677490234  acc test= 87.82400512695312
idx= 250  nbgood= 6151
idx= 250  nbgood= 5615
** batch:  250  acc train= 98.06400299072266  acc test= 89.40800476074219
idx= 250  nbgood= 6227
idx= 250  nbgood= 5605
**** epoch 3  acc train= 99.21600341796875  acc test= 89.52000427246094
idx= 250  nbgood= 6225
idx= 250  nbgood= 5623
** batch:  250  acc train= 99.27200317382812  acc test= 89.68000793457031
idx= 250  nbgood= 6248
idx= 250  nbgood= 5635
**** epoch 4  acc train= 99.60000610351562  acc test= 89.84800720214844
idx= 250  nbgood= 6239
idx= 250  nbgood= 5593
**